In [3]:
# Baseline LightGBM Pipeline for Kaggle Playground Series S6E1
# Exam Score Prediction (RMSE)

import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import lightgbm as lgb

In [4]:
# --------------------------------------------------
# 1. Load data
# --------------------------------------------------
train = pd.read_csv('/kaggle/input/playground-series-s6e1/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e1/test.csv')

TARGET = 'exam_score'
ID_COL = 'id'

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET]
X_test = test.drop(columns=[ID_COL])

In [5]:
# --------------------------------------------------
# 2. Feature groups
# --------------------------------------------------
num_features = [
    'age',
    'study_hours',
    'class_attendance',
    'sleep_hours'
]

cat_features = [
    'gender',
    'course',
    'sleep_quality',
    'study_method',
    'facility_rating',
    'exam_difficulty',
    'internet_access'
]

In [6]:
# --------------------------------------------------
# 3. Preprocessing
# --------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse=False), cat_features)
    ]
)

In [7]:
# --------------------------------------------------
# 4. LightGBM model (baseline)
# --------------------------------------------------
lgbm = lgb.LGBMRegressor(
    objective='regression',
    metric='rmse',
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [8]:
# --------------------------------------------------
# 5. Pipeline
# --------------------------------------------------
pipeline = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('model', lgbm)
    ]
)

In [9]:
# --------------------------------------------------
# 6. Cross-validation
# --------------------------------------------------
cv = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_val)
    rmse = mean_squared_error(y_val, preds, squared=False)

    rmse_scores.append(rmse)
    print(f'Fold {fold + 1} RMSE: {rmse:.5f}')

print(f'Mean CV RMSE: {np.mean(rmse_scores):.5f}')

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017243 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 622
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 30
[LightGBM] [Info] Start training from score 62.482335
Fold 1 RMSE: 8.75677


/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014759 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 622
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 30
[LightGBM] [Info] Start training from score 62.502155
Fold 2 RMSE: 8.76076


/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015533 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 30
[LightGBM] [Info] Start training from score 62.523833
Fold 3 RMSE: 8.75511


/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.032411 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 623
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 30
[LightGBM] [Info] Start training from score 62.522425
Fold 4 RMSE: 8.77402


/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014948 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 30
[LightGBM] [Info] Start training from score 62.502613
Fold 5 RMSE: 8.78986
Mean CV RMSE: 8.76731


In [10]:
# --------------------------------------------------
# 7. Train on full data & predict test
# --------------------------------------------------
pipeline.fit(X, y)
test_preds = pipeline.predict(X_test)

submission = pd.DataFrame({
    'id': test[ID_COL],
    'exam_score': test_preds
})

submission.to_csv('submission.csv', index=False)
print('submission.csv saved')

/usr/local/lib/python3.11/dist-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019049 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 621
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 30
[LightGBM] [Info] Start training from score 62.506672
submission.csv saved
